In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DataFrames").getOrCreate()


In [0]:
# Emp Data & Schema

emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [0]:

# Create emp DataFrame

emp = spark.createDataFrame(data=emp_data, schema=emp_schema)

In [0]:
# Show emp dataframe (ACTION)

emp.show()

In [0]:
from pyspark.sql.functions import col,split
# split name column into two first_name and last_name
emp_name_split = emp.select('employee_id','department_id','name','age','gender','salary','hire_date').withColumn('first_name',split(col('name'),' ')[0]).withColumn('last_name',split(col('name'),' ')[1])

In [0]:
emp_name_split = emp_name_split.drop('name')
emp_name_split.show()

In [0]:
# cast type
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType,DateType
emp_name_split = emp_name_split.select(col('employee_id').cast(IntegerType()),col('department_id').cast(IntegerType()),'first_name','last_name',col('age').cast(IntegerType()) ,'gender',col('salary').cast(IntegerType()),col('hire_date').cast(DateType()))
emp_name_split.show()

In [0]:
# Add new column from existing one
from pyspark.sql.functions import *
emp_name_split = emp_name_split.withColumn('Experience',round(months_between(current_date(),col('hire_date'))/lit(12),2))
emp_name_split.show()

In [0]:
# case when
from pyspark.sql.functions import when, col
emp_df = emp_name_split.select('employee_id',"department_id","first_name","last_name","age","gender",'salary','hire_date','Experience')
emp_df = emp_df.withColumn('gender',
                           when(col('gender') == 'Male','M').
                           when(col('gender') == 'Female','F').
                           otherwise('O'))
emp_df.show()

In [0]:
from pyspark.sql.functions import col, sum, avg, count,max
# Group By, Order By and Where

emp_female_df = emp_df.where(col('gender') == 'F')
emp_male_df = emp_df.where(col('gender') == 'M')
emp_female_df.show()
emp_male_df.show()

emp_female_df = emp_female_df.orderBy(col('salary').desc())
emp_male_df = emp_male_df.orderBy(col('salary').desc())
emp_female_df.show()
emp_male_df.show()

emp_group_by_department = emp_df.groupBy('department_id').agg(sum('salary').alias('Total Salary of Department'))
emp_group_by_department.show()

emp_group_by_department_and_gender = emp_df.groupBy('department_id','gender').agg(count('employee_id').alias('# of Employee'),avg('salary').alias('Sum Of Employee'))
emp_group_by_department_and_gender.show()

In [0]:
# Window Functions
from pyspark.sql.window  import Window
import pyspark.sql.functions as F

# get max salary by gender and department
emp_salary_df = emp_df.withColumn('Rank',F.rank().over(Window.partitionBy(col('department_id')).orderBy(col('salary').desc())))
emp_salary_df= emp_salary_df.select('department_id','first_name','last_name','gender','salary','Experience').where(col('Rank') == 1)
emp_salary_df.show()